STEP 1

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")
warnings.filterwarnings("ignore", category=RuntimeWarning)# ============================================================
# CONFIG
# ============================================================
DATA_FILE = "resources/M1045_MonthlyCrimeDashboard_TNOCrimeData.xlsx"


# ============================================================
# SECTION 1 — DATA PIPELINE
# ============================================================
def load_and_clean_data(file_path: str) -> pd.DataFrame:
    df = pd.read_excel(file_path)
    df = df.rename(columns={
    "Area name": "Area_name",
    "Offence Group": "Offence_Group"
    })

    df["Area_name"] = df["Area_name"].str.strip()
    df = df[df["Area Type"] == "Borough"]


    columns_to_drop = ["Borough_SNT", "Area code", "FY_FYIndex", "Refresh Date"]
    df = df.drop(columns=columns_to_drop, errors="ignore")

    return df


def aggregate_and_compute_success_ratio(df: pd.DataFrame) -> pd.DataFrame:
    grouped = (
        df.groupby(["Month_Year", "Area_name", "Offence_Group", "Measure"], as_index=False)["Count"]
          .sum()
    )

    pivot = grouped.pivot_table(
        index=["Month_Year", "Area_name", "Offence_Group"],
        columns="Measure",
        values="Count",
        fill_value=0
    ).reset_index()

    if "Offences" not in pivot.columns:
        pivot["Offences"] = 0
    if "Positive Outcomes" not in pivot.columns:
        pivot["Positive Outcomes"] = 0

    totals = (
        pivot.groupby(["Month_Year", "Area_name"], as_index=False)[["Offences", "Positive Outcomes"]]
             .sum()
    )
    totals["Offence_Group"] = "Total"

    cols = ["Month_Year", "Area_name", "Offence_Group", "Offences", "Positive Outcomes"]
    pivot = pivot[cols]
    totals = totals[cols]

    pivot_with_totals = pd.concat([pivot, totals], ignore_index=True)

    pivot_with_totals["Success Ratio"] = (
        pivot_with_totals["Positive Outcomes"] / pivot_with_totals["Offences"]
    ).fillna(0)

    return pivot_with_totals

STEP 2

In [ ]:
# ============================================================
# SECTION 2 — USER INPUT HELPERS
# ============================================================
def choose_borough(df: pd.DataFrame, prompt="Choose a borough: "):
    boroughs = sorted(df["Area_name"].unique())
    print("\nAvailable boroughs:")
    for i, b in enumerate(boroughs, 1):
        print(f"{i}. {b}")

    while True:
        choice = input(prompt)
        if choice.isdigit():
            idx = int(choice)
            if 1 <= idx <= len(boroughs):
                return boroughs[idx - 1]
        if choice in boroughs:
            return choice
        print("Invalid choice, try again.")


def choose_offence_group(df: pd.DataFrame):
    groups = sorted(df["Offence_Group"].unique())
    print("\nAvailable offence groups:")
    for i, g in enumerate(groups, 1):
        print(f"{i}. {g}")

    while True:
        choice = input("Choose an offence group: ")
        if choice.isdigit():
            idx = int(choice)
            if 1 <= idx <= len(groups):
                return groups[idx - 1]
        if choice in groups:
            return choice
        print("Invalid choice, try again.")


def choose_measure():
    options = ["Offences", "Positive Outcomes", "Success Ratio"]
    print("\nMeasures:")
    for i, m in enumerate(options, 1):
        print(f"{i}. {m}")

    while True:
        choice = input("Choose a measure (1-3): ")
        if choice.isdigit():
            idx = int(choice)
            if 1 <= idx <= len(options):
                return options[idx - 1]
        print("Invalid choice, try again.")

STEP 3

In [ ]:
#============================================================
# SECTION 3 — PLOTTING FUNCTIONS
# ============================================================
def plot_single_borough(df):
    borough = choose_borough(df, "Choose a borough (single): ")
    offence_group = choose_offence_group(df)
    measure = choose_measure()

    subset = df[
        (df["Area_name"] == borough) &
        (df["Offence_Group"] == offence_group)
    ].sort_values("Month_Year")

    if subset.empty:
        print("No data for that combination.")
        return

    plt.figure(figsize=(10, 5))
    sns.scatterplot(data=subset, x="Month_Year", y=measure)
    plt.title(f"{measure} over time — {borough}, {offence_group}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


def plot_two_boroughs(df):
    print("\nFirst borough:")
    b1 = choose_borough(df)
    print("\nSecond borough:")
    b2 = choose_borough(df)

    offence_group = choose_offence_group(df)
    measure = choose_measure()

    s1 = df[(df["Area_name"] == b1) & (df["Offence_Group"] == offence_group)][["Month_Year", measure]]
    s2 = df[(df["Area_name"] == b2) & (df["Offence_Group"] == offence_group)][["Month_Year", measure]]

    merged = pd.merge(
        s1.rename(columns={measure: b1}),
        s2.rename(columns={measure: b2}),
        on="Month_Year",
        how="inner"
    ).sort_values("Month_Year")

    if merged.empty:
        print("No overlapping data.")
        return

    corr = merged[b1].corr(merged[b2])
    print(f"\nCorrelation ({measure}) between {b1} and {b2}: {corr:.4f}")

    plt.figure(figsize=(10, 5))
    sns.scatterplot(data=merged, x="Month_Year", y=b1, label=b1)
    sns.scatterplot(data=merged, x="Month_Year", y=b2, label=b2)
    plt.title(f"{measure} — {b1} vs {b2}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


def plot_two_boroughs_demeaned(df):
    print("\nFirst borough:")
    b1 = choose_borough(df)
    print("\nSecond borough:")
    b2 = choose_borough(df)

    offence_group = choose_offence_group(df)
    measure = choose_measure()

    s1 = df[(df["Area_name"] == b1) & (df["Offence Group"] == offence_group)][["Month_Year", measure]].copy()
    s2 = df[(df["Area_name"] == b2) & (df["Offence Group"] == offence_group)][["Month_Year", measure]].copy()

    if s1.empty or s2.empty:
        print("No data.")
        return

    s1[measure] -= s1[measure].mean()
    s2[measure] -= s2[measure].mean()

    merged = pd.merge(
        s1.rename(columns={measure: f"{b1}_d"}),
        s2.rename(columns={measure: f"{b2}_d"}),
        on="Month_Year",
        how="inner"
    ).sort_values("Month_Year")

    corr = merged[f"{b1}_d"].corr(merged[f"{b2}_d"])
    print(f"\nDemeaned correlation ({measure}) between {b1} and {b2}: {corr:.4f}")

    plt.figure(figsize=(10, 5))
    sns.scatterplot(data=merged, x="Month_Year", y=f"{b1}_d", label=f"{b1} demeaned")
    sns.scatterplot(data=merged, x="Month_Year", y=f"{b2}_d", label=f"{b2} demeaned")
    plt.title(f"Demeaned {measure} — {b1} vs {b2}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


STEP 4

In [ ]:
# ============================================================
# SECTION 4 — CORRELATION ENGINE
# ============================================================
def compute_all_correlations(df, base_borough):
    total_df = df[df["Offence Group"] == "Total"].copy()
    total_df["SR_d"] = total_df.groupby("Area_name")["Success Ratio"].transform(lambda x: x - x.mean())

    base = total_df[total_df["Area_name"] == base_borough][["Month_Year", "SR_d"]]
    base = base.rename(columns={"SR_d": "Base"})

    results = []

    for borough in sorted(total_df["Area_name"].unique()):
        if borough == base_borough:
            continue

        other = total_df[total_df["Area_name"] == borough][["Month_Year", "SR_d"]]
        other = other.rename(columns={"SR_d": "Other"})

        merged = pd.merge(base, other, on="Month_Year", how="inner")

        if merged.empty:
            corr = np.nan
        else:
            corr = merged["Base"].corr(merged["Other"])

        results.append({"Area_name": borough, "Correlation": corr})

    corr_df = pd.DataFrame(results).sort_values("Correlation", ascending=False)
    print("\nCorrelation of demeaned success ratio (Total) vs all boroughs:")
    print(corr_df.to_string(index=False))
    return corr_df

STEP 5

In [ ]:
# ============================================================
# SECTION 5 — CUBIC REGRESSION MODEL
# ============================================================
def fit_and_plot_cubic(df, borough):
    total_df = df[(df["Offence_Group"] == "Total") & (df["Area_name"] == borough)].copy()

    if total_df.empty:
        print("No data.")
        return

    total_df = total_df.sort_values("Month_Year")
    total_df["SR_d"] = total_df["Success Ratio"] - total_df["Success Ratio"].mean()
    total_df["t"] = total_df["Month_Year"].map(lambda x: x.toordinal())

    X = total_df[["t"]].values
    y = total_df["SR_d"].values

    n = len(total_df)
    split = int(0.75 * n)

    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    poly = PolynomialFeatures(degree=3, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)

    model = LinearRegression()
    model.fit(X_train_poly, y_train)

    X_full_poly = poly.transform(X)
    y_pred = model.predict(X_full_poly)

    plt.figure(figsize=(10, 5))
    plt.scatter(total_df["Month_Year"], y, label="Demeaned Success Ratio")
    plt.plot(total_df["Month_Year"], y_pred, color="red", label="Cubic Fit")
    plt.title(f"Cubic Fit — {borough}")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()

    mse = np.mean((y_test - model.predict(X_test_poly)) ** 2)
    print(f"\nTest MSE for cubic model ({borough}): {mse:.6f}")

STEP 6

In [ ]:
# ============================================================
# TESTS
# ============================================================
# TEST 1 — DATA LOADING & CLEANING
# ============================================================
def test_data_loading():
    print("\n=== TEST 1: DATA LOADING & CLEANING ===")
    df_raw = load_and_clean_data(DATA_FILE)
    

    print("\nFirst 5 rows:")
    print(df_raw.head())

    print("\nUnique Area Types:")
    print(df_raw["Area Type"].unique())

    print("\nColumns after cleaning:")
    print(df_raw.columns)

    print("\nEXPECTED:")
    print("- Only 'Borough' should appear in Area Type")
    print("- Columns Borough_SNT, Area code, FY_FYIndex, Refresh Date should be removed")

    return df_raw


# ============================================================
# TEST 2 — AGGREGATION & SUCCESS RATIO
# ============================================================
def test_aggregation(df_raw):
    print("\n=== TEST 2: AGGREGATION & SUCCESS RATIO ===")
    df = aggregate_and_compute_success_ratio(df_raw)

    print("\nSample TOTAL rows:")
    print(df[df["Offence_Group"] == "Total"].head())

    print("\nEXPECTED:")
    print("- Each borough/date should have a 'Total' row")
    print("- Success Ratio column should exist")

    return df


# ============================================================
# TEST 3 — SINGLE BOROUGH SCATTERPLOT
# ============================================================
def test_single_borough_plot(df):
    print("\n=== TEST 3: SINGLE BOROUGH SCATTERPLOT ===")
    print("Run the function and choose:")
    print("- Borough: Camden")
    print("- Offence Group: Violence Against the Person")
    print("- Measure: Success Ratio")
    print("\nA scatterplot should appear.\n")

    plot_single_borough(df)


# ============================================================
# TEST 4 — TWO BOROUGH SCATTERPLOT + CORRELATION
# ============================================================
def test_two_borough_plot(df):
    print("\n=== TEST 4: TWO BOROUGH SCATTERPLOT + CORRELATION ===")
    print("Run the function and choose:")
    print("- Boroughs: Camden & Croydon")
    print("- Offence Group: Total")
    print("- Measure: Success Ratio")
    print("\nTwo scatter series should appear + correlation printed.\n")

    plot_two_boroughs(df)


# ============================================================
# TEST 5 — DEMEANED SCATTERPLOT
# ============================================================
def test_demeaned_plot(df):
    print("\n=== TEST 5: DEMEANED SCATTERPLOT ===")
    print("Run the function and choose:")
    print("- Boroughs: Camden & Croydon")
    print("- Offence Group: Total")
    print("- Measure: Success Ratio")
    print("\nPlot should be centered around zero + correlation printed.\n")

    plot_two_boroughs_demeaned(df)


# ============================================================
# TEST 6 — CORRELATION ENGINE (ALL BOROUGHS)
# ============================================================
def test_correlation_engine(df):
    print("\n=== TEST 6: CORRELATION ENGINE ===")
    print("Choose a borough to compute correlations against all others.")

    base = choose_borough(df, "Choose base borough for correlation test: ")
    corr_df = compute_all_correlations(df, base)

    print("\nEXPECTED:")
    print("- A sorted list of boroughs with correlation values")

    return corr_df


def test_cubic_regression(df):
    print("\n=== TEST 8: CUBIC REGRESSION MODEL ===")
    print("Choose a borough for cubic regression.")

    b = choose_borough(df, "Choose borough for cubic model: ")
    fit_and_plot_cubic(df, b)

    print("\nEXPECTED:")
    print("- Scatterplot + cubic curve")
    print("- Test MSE printed")


def run_all_tests(df_raw=None, df=None):
    print("\n========================================")
    print(" RUNNING FULL TEST SUITE")
    print("========================================")

    # Load + aggregate fresh data if not provided
    if df_raw is None:
        df_raw = test_data_loading()
    if df is None:
        df = test_aggregation(df_raw)

    # Interactive tests
    test_single_borough_plot(df)
    test_two_borough_plot(df)
    test_demeaned_plot(df)

    # Correlation engine
    corr_df = test_correlation_engine(df)

    # Cubic regression
    test_cubic_regression(df)

    print("\n========================================")
    print(" ALL TESTS COMPLETED")
    print("========================================")



# ============================================================
#  MAIN MENU
# ============================================================
def main():
    print("Loading and preparing data...")
    df_raw = load_and_clean_data(DATA_FILE)
    df = aggregate_and_compute_success_ratio(df_raw)

    while True:
        print("\n==================== MENU ====================")
        print("1. Plot single borough")
        print("2. Plot two boroughs")
        print("3. Plot two boroughs (demeaned)")
        print("4. Compute correlations vs all boroughs (demeaned, Total)")
        print("5. Fit cubic regression model (demeaned, Total)")
        print("6. Exit")
        print("7. Run full testing suite")

        choice = input("Choose an option (1-7): ")

        if choice == "1":
            plot_single_borough(df)
        elif choice == "2":
            plot_two_boroughs(df)
        elif choice == "3":
            plot_two_boroughs_demeaned(df)
        elif choice == "4":
            base = choose_borough(df, "Choose base borough: ")
            compute_all_correlations(df, base)
        elif choice == "5":
            b = choose_borough(df, "Choose borough for cubic model: ")
            fit_and_plot_cubic(df, b)
        elif choice == "7":
            print("\nRunning full testing suite...")
            run_all_tests()
        elif choice == "6":
            print("Exiting.")
            break
        else:
            print("Invalid choice.")


# ============================================================
# ENTRY POINT
# ============================================================
if __name__ == "__main__":
    main()


STEP 7

In [ ]:
# ============================================================
# — EXPORT CLEANED + AGGREGATED DATASET
# ============================================================

# 1. Load and clean raw data
df_raw = load_and_clean_data(DATA_FILE)

# 2. Aggregate and compute success ratios
df_final = aggregate_and_compute_success_ratio(df_raw)

df_final = df_final.rename(columns={
    "Positive Outcomes": "Positive_Outcomes",
    "Success Ratio": "Success_Ratio"
})

# 3. Optional: sort for readability
df_final = df_final.sort_values(["Area_name", "Month_Year"])

# 4. Export to Excel
output_path = "final_cleaned_crime_data.xlsx"
df_final.to_excel(output_path, index=False)

print(f"Final cleaned dataset exported to: {output_path}")
print("Rows:", len(df_final))
print("Columns:", list(df_final.columns))
